<a href="https://colab.research.google.com/github/j019/Practical-Machine-Learning/blob/main/Day7/CaseStudy_Early_Wakeup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Case Study : Early Wakeup Health
- Dataset : https://www.kaggle.com/datasets/nalisha/early-wakeup-health-and-lifestyle-dataset

In [55]:
import pandas as pd
df = pd.read_csv('/content/early_wakeup_health_dataset.csv')

In [56]:
df.shape

(10000, 64)

In [57]:
df.isna().sum().sum()

np.int64(4662)

In [58]:
df.isna().sum()

,0
Person_ID,0
Age,0
Gender,0
Height_cm,0
Weight_kg,0
...,...
Health_Score,0
Fitness_Level,0
Healthy_Aging_Score,0
Wellness_Category,0


In [59]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 64 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Person_ID                       10000 non-null  object 
 1   Age                             10000 non-null  int64  
 2   Gender                          10000 non-null  object 
 3   Height_cm                       10000 non-null  float64
 4   Weight_kg                       10000 non-null  float64
 5   BMI                             10000 non-null  float64
 6   Country                         10000 non-null  object 
 7   Occupation                      10000 non-null  object 
 8   Marital_Status                  10000 non-null  object 
 9   Wake_Up_Time                    10000 non-null  object 
 10  Sleep_Time                      10000 non-null  object 
 11  Sleep_Duration_Hours            10000 non-null  float64
 12  Sleep_Quality_Score             1

## drop columns --> personId, height, Weight, Gym_Member,Earlywaker



### Drop Target Columns -->
 - Obesity_risk, Hypertension, sleepdisorder, Cardiovascular,
 - HealthyAgeing, Wellness, Depression_risk_score, Depression_risk

### Target Columns-->
- Fitness_level --> Multi Class Classification task

- Country --> check distribution --> remove less occurence --> Done --> No need to remove

### -->
* Productivity_Score --> to be discussed


In [60]:
df['Country'].value_counts()

,count
Country,
USA,1468
India,1206
UK,798
Canada,763
Japan,667
Germany,623
Pakistan,598
Australia,574
Brazil,572


In [61]:
# Drop Columns(irrelevant & target Columns)
df.drop(columns=['Obesity_Risk',
'Hypertension_Risk',
'Diabetes_Risk',
'Cardiovascular_Risk',
'Sleep_Disorder_Risk',
'Health_Score',
'Healthy_Aging_Score',
'Wellness_Category',
'Early_Waker',
'Depression_Risk_Score',
'Gym_Member',
'Height_cm',
'Weight_kg',
'Person_ID'],inplace=True)



In [62]:
df.shape

(10000, 50)

## Convert Date Time to Category (Wake Up Time & Sleep Time)
### Preprocessing on Wake up Time & Sleep Time
#### Wake up Time --> Convert to categorical
- 4am - 6am --> Early Morning --> 0
- 6am - 8am --> Morning Time --> 1
- 8am - 10am --> Late --> 2
- 10am - 11am --> Very Late -->3

### Sleep Time --> Convert to categorical
- 6pm - 8pm --> Outlier --> 5
- 8pm - 10pm --> Early --> 0
- 10pm - 12am --> On-time --> 1
- 0am - 3am --> Late --> 2
- 3am - 7am --> Very Late --> 4




In [63]:
df['Wake_Up_Time'] = pd.to_datetime(df['Wake_Up_Time'])
df['Sleep_Time'] = pd.to_datetime(df['Sleep_Time'])

/tmp/ipykernel_1848/214474565.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Wake_Up_Time'] = pd.to_datetime(df['Wake_Up_Time'])
/tmp/ipykernel_1848/214474565.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Sleep_Time'] = pd.to_datetime(df['Sleep_Time'])


In [64]:
df['Wake_Up_Time'].head()

,Wake_Up_Time
0,2026-06-27 08:59:00
1,2026-06-27 06:35:00
2,2026-06-27 07:31:00
3,2026-06-27 08:36:00
4,2026-06-27 07:07:00


In [65]:
df['WT_cat'] = 0 #WakeUp_time -->categorical

In [66]:
df.loc[(df['Wake_Up_Time'] >= "2026-06-27 04:00:00") & (df['Wake_Up_Time'] < "2026-06-27 06:00:00"), "WT_cat" ] = 0
df.loc[(df['Wake_Up_Time'] >= "2026-06-27 06:00:00") & (df['Wake_Up_Time'] < "2026-06-27 08:00:00"), "WT_cat" ] = 1
df.loc[(df['Wake_Up_Time'] >= "2026-06-27 08:00:00") & (df['Wake_Up_Time'] < "2026-06-27 10:00:00"), "WT_cat" ] = 2
df.loc[(df['Wake_Up_Time'] >= "2026-06-27 10:00:00") & (df['Wake_Up_Time'] <= "2026-06-27 11:00:00"), "WT_cat" ] = 3

In [67]:
df['WT_cat'].unique()

array([2, 1, 0, 3])

In [68]:
df['Sleep_Time'].head()

,Sleep_Time
0,2026-06-27 01:29:00
1,2026-06-27 02:25:00
2,2026-06-27 02:24:00
3,2026-06-27 01:19:00
4,2026-06-27 01:20:00


In [69]:
df['Sleep_Time'].sort_values()

,Sleep_Time
7302,2026-06-27 00:00:00
9058,2026-06-27 00:00:00
7792,2026-06-27 00:00:00
7692,2026-06-27 00:00:00
5933,2026-06-27 00:00:00
...,...
7273,2026-06-27 23:59:00
5927,2026-06-27 23:59:00
7115,2026-06-27 23:59:00
2006,2026-06-27 23:59:00


In [70]:
df['ST_cat'] = 0

In [71]:
df.loc[(df['Sleep_Time'] >= "2026-06-27 20:00:00") & (df['Sleep_Time'] < "2026-06-27 22:00:00"), "ST_cat" ] = 0
df.loc[(df['Sleep_Time'] >= "2026-06-27 22:00:00") & (df['Sleep_Time'] <= "2026-06-27 23:59:00"), "ST_cat" ] = 1
df.loc[(df['Sleep_Time'] >= "2026-06-27 00:00:00") & (df['Sleep_Time'] < "2026-06-27 03:00:00"), "ST_cat" ] = 2
df.loc[(df['Sleep_Time'] >= "2026-06-27 03:00:00") & (df['Sleep_Time'] < "2026-06-27 07:00:00"), "ST_cat" ] = 3
## Outlier
df.loc[(df['Sleep_Time'] >= "2026-06-27 18:00:00") & (df['Sleep_Time'] < "2026-06-27 20:00:00"), "ST_cat" ] = 4

In [72]:
df['ST_cat'].unique()

array([2, 1, 0, 4, 3])

In [73]:
df.drop(columns=['Wake_Up_Time', 'Sleep_Time'],inplace=True)

In [74]:
df.shape

(10000, 50)

In [75]:
df.columns

Index(['Age', 'Gender', 'BMI', 'Country', 'Occupation', 'Marital_Status',
       'Sleep_Duration_Hours', 'Sleep_Quality_Score',
       'Number_of_Night_Awakenings', 'Weekend_Sleep_Difference_Hours',
       'Nap_Frequency_Per_Week', 'Screen_Time_Before_Bed_Hours',
       'Exercise_Frequency_Per_Week', 'Exercise_Duration_Minutes',
       'Exercise_Type', 'Daily_Steps', 'Morning_Workout', 'Workout_Intensity',
       'Daily_Calorie_Intake', 'Water_Intake_Liters', 'Fruit_Intake_Per_Day',
       'Vegetable_Intake_Per_Day', 'Protein_Intake_Grams',
       'Sugary_Drinks_Per_Week', 'Fast_Food_Meals_Per_Week',
       'Breakfast_Regularity_Score', 'Smoking_Status', 'Alcohol_Consumption',
       'Stress_Level', 'Working_Hours_Per_Day', 'Sitting_Hours_Per_Day',
       'Outdoor_Time_Hours', 'Social_Interaction_Score', 'Meditation_Practice',
       'Resting_Heart_Rate', 'Systolic_BP', 'Diastolic_BP',
       'Cholesterol_Level', 'Blood_Sugar_Level', 'Energy_Level_Score',
       'Fatigue_Level_Score', 

## Fill missing Values

In [77]:
df.isnull().sum() #Exercise_Type,Workout_Intensity,Alcohol_Consumption

,0
Age,0
Gender,0
BMI,0
Country,0
Occupation,0
Marital_Status,0
Sleep_Duration_Hours,0
Sleep_Quality_Score,0
Number_of_Night_Awakenings,0
Weekend_Sleep_Difference_Hours,0


## According to Statisticians ,When we replace it with central tendency then the data is balanced

- Be careful while replacing with mode use .iloc[0] --> in case of multiple modal

In [78]:
df.loc[df['Alcohol_Consumption'].isna() == True, 'Alcohol_Consumption'] = 0

In [79]:
df[['Exercise_Type','Workout_Intensity']].mode().iloc[0]

,0
Exercise_Type,Weight Training
Workout_Intensity,Moderate


In [80]:
#fill_values = df[['Exercise_Type','Workout_Intensity','Alcohol_Consumption']].mode().iloc[0]
#df[['Exercise_Type','Workout_Intensity','Alcohol_Consumption']].fillna(fill_values,inplace=True)

In [81]:
fill_values = df.mode().iloc[0]
df.fillna(fill_values,inplace=True)

In [82]:
df.isnull().sum().sum()

np.int64(0)

## Handle Ordinal columns (label_encoding):
- Workout_Intensity
- Smoking_Status
- Alcohol_Consumption

In [83]:
df['Workout_Intensity'].unique()

array(['Low', 'Moderate', 'High'], dtype=object)

In [84]:
df['Smoking_Status'].unique()

array(['Former', 'Current', 'Never'], dtype=object)

In [85]:
df['Alcohol_Consumption'].unique()

array(['Moderate', 0, 'Light', 'Heavy'], dtype=object)

In [94]:
label_map={
    'Workout_Intensity':{
        'Low':0,
        'Moderate':1,
        'High':2
    },
    'Smoking_Status':{
        'Former':1, 'Current':2, 'Never':0
    },
    'Alcohol_Consumption':{
        'Light':1,
        'Moderate':2,
        'Heavy':3
    }
}

In [95]:
df.replace(label_map,inplace=True)

/tmp/ipykernel_1848/1715271054.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace(label_map,inplace=True)


In [96]:
df[df.columns[df.nunique() > 15]].dtypes

,0
Age,int64
BMI,float64
Sleep_Duration_Hours,float64
Sleep_Quality_Score,float64
Weekend_Sleep_Difference_Hours,float64
Screen_Time_Before_Bed_Hours,float64
Exercise_Duration_Minutes,int64
Daily_Steps,int64
Daily_Calorie_Intake,int64
Water_Intake_Liters,float64


In [97]:
df['Workout_Intensity'].unique()

array([0, 1, 2])

In [98]:
df['Alcohol_Consumption'].unique()

array([2, 0, 1, 3])

In [99]:
df['Smoking_Status'].unique()

array([1, 2, 0])

In [100]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 50 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Age                             10000 non-null  int64  
 1   Gender                          10000 non-null  object 
 2   BMI                             10000 non-null  float64
 3   Country                         10000 non-null  object 
 4   Occupation                      10000 non-null  object 
 5   Marital_Status                  10000 non-null  object 
 6   Sleep_Duration_Hours            10000 non-null  float64
 7   Sleep_Quality_Score             10000 non-null  float64
 8   Number_of_Night_Awakenings      10000 non-null  int64  
 9   Weekend_Sleep_Difference_Hours  10000 non-null  float64
 10  Nap_Frequency_Per_Week          10000 non-null  int64  
 11  Screen_Time_Before_Bed_Hours    10000 non-null  float64
 12  Exercise_Frequency_Per_Week     1

# X & Y Split

- target : fitness level

In [101]:
X = df.drop(columns=['Fitness_Level'])
Y = df['Fitness_Level']

X.shape, Y.shape

((10000, 49), (10000,))

In [102]:
Y.unique()

array(['Excellent', 'Good', 'Average', 'Poor'], dtype=object)

In [103]:
label_map = {'Excellent':3, 'Good':2, 'Average':1, 'Poor':0}
Y.replace(label_map,inplace=True)

/tmp/ipykernel_1848/1394058508.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Y.replace(label_map,inplace=True)


In [104]:
Y.unique()

array([3, 2, 1, 0])

In [105]:
X_ohe = pd.get_dummies(X)
X_ohe.shape

(10000, 91)

In [106]:
X_ohe.columns

Index(['Age', 'BMI', 'Sleep_Duration_Hours', 'Sleep_Quality_Score',
       'Number_of_Night_Awakenings', 'Weekend_Sleep_Difference_Hours',
       'Nap_Frequency_Per_Week', 'Screen_Time_Before_Bed_Hours',
       'Exercise_Frequency_Per_Week', 'Exercise_Duration_Minutes',
       'Daily_Steps', 'Workout_Intensity', 'Daily_Calorie_Intake',
       'Water_Intake_Liters', 'Fruit_Intake_Per_Day',
       'Vegetable_Intake_Per_Day', 'Protein_Intake_Grams',
       'Sugary_Drinks_Per_Week', 'Fast_Food_Meals_Per_Week',
       'Breakfast_Regularity_Score', 'Smoking_Status', 'Alcohol_Consumption',
       'Stress_Level', 'Working_Hours_Per_Day', 'Sitting_Hours_Per_Day',
       'Outdoor_Time_Hours', 'Social_Interaction_Score', 'Resting_Heart_Rate',
       'Systolic_BP', 'Diastolic_BP', 'Cholesterol_Level', 'Blood_Sugar_Level',
       'Energy_Level_Score', 'Fatigue_Level_Score', 'Immune_Health_Score',
       'Mood_Score', 'Anxiety_Score', 'Productivity_Score',
       'Focus_Concentration_Score', 'Life_S

In [111]:
cat_col = X_ohe.columns[X_ohe.nunique() <= 15]
cat_col.shape

(61,)

In [112]:
cont_col = X_ohe.columns[X_ohe.nunique() > 15]
cont_col.shape

(30,)

## Train-test Split

In [114]:
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X_ohe, Y, test_size=0.3, random_state=7,stratify=Y)
X_train.shape, X_test.shape, Y_train.shape, Y_test.shape

((7000, 91), (3000, 91), (7000,), (3000,))

In [115]:
from sklearn.preprocessing import RobustScaler
rc = RobustScaler()
X_train[cont_col] = rc.fit_transform(X_train[cont_col])
X_test[cont_col] = rc.transform(X_test[cont_col])

In [116]:
X_train[cont_col].shape, X_test[cont_col].shape

((7000, 30), (3000, 30))

## Apply Outlier Imputation Using IQR

In [118]:
def outlier_impute_iqr(df,col):
  df1 = df.copy()
  print("Column --> ",col)
  q1,q3 = df1[col].quantile([0.25,0.75])
  iqr = q3-q1
  lb = q1 - 1.5*iqr
  ub = q3 + 1.5*iqr
  df1.loc[(df1[col] < lb),col] = lb
  df1.loc[(df1[col] > ub),col] = ub
  #df1.loc[(df1[col]UB),col] = UB
  print(lb, ub, df1[col].min(), df1[col].max())
  return df1

In [119]:
X_train_io = X_train.copy()
for col in cont_col:
  X_train_io = outlier_impute_iqr(X_train_io,col)

Column -->  Age
-1.9642857142857144 2.0357142857142856 -0.8928571428571429 1.3214285714285714
Column -->  BMI
-2.0089153046062407 1.9910846953937595 -1.4257057949479945 1.9910846953937595
Column -->  Sleep_Duration_Hours
-2.0033557046979866 1.9966442953020134 -2.0033557046979866 1.8523489932885902
Column -->  Sleep_Quality_Score
-2.0 2.0 -2.0 1.7272727272727282
Column -->  Weekend_Sleep_Difference_Hours
-2.0092592592592595 1.9907407407407407 -1.8888888888888888 1.9907407407407407
Column -->  Screen_Time_Before_Bed_Hours
-1.8664122137404582 2.133587786259542 -0.618320610687023 2.133587786259542
Column -->  Exercise_Duration_Minutes
-2.0 2.0 -1.6071428571428572 2.0
Column -->  Daily_Steps
-2.025921583952065 1.9740784160479354 -1.9954409274456169 1.9740784160479354
Column -->  Daily_Calorie_Intake
-1.9973230220107079 2.0026769779892923 -1.9973230220107079 2.0026769779892923
Column -->  Water_Intake_Liters
-1.9999999999999996 2.0000000000000004 -1.9999999999999996 2.0000000000000004
Column